# Earthquake Detection — Interactive Notebook
Run cells top-to-bottom. Edit the **Configuration** cell to tune parameters.

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

## Configuration

In [ ]:
CSV_PATH         = "accelerometer_data.csv"
BP_LOW_HZ        = 1.0
BP_HIGH_HZ       = 20.0
STA_WINDOW_S     = 0.5
LTA_WINDOW_S     = 10.0
STA_LTA_THRESH   = 3.0
AMP_SIGMA_THRESH = 4.0
MERGE_GAP_S      = 3.0
QUIET_GUARD_S    = 8.0

## Stage 1 — Load & compute magnitude

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip().str.lower()

time_col = next((c for c in df.columns if 'time' in c), None)
if time_col:
    df[time_col] = pd.to_datetime(df[time_col])
    t_s = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds().values
    fs  = round(1.0 / np.median(np.diff(t_s)))
    timestamps = df[time_col]
else:
    fs = 100
    t_s = np.arange(len(df)) / fs
    timestamps = None

mag = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2).values
print(f"{len(mag):,} samples | fs={fs} Hz | duration={t_s[-1]:.1f} s")

## Stage 2 — Pre-processing

In [ ]:
sig = mag - np.mean(mag)
nyq = fs / 2.0
sos = butter(4, [BP_LOW_HZ/nyq, min(BP_HIGH_HZ/nyq, 0.99)], btype='band', output='sos')
filtered = sosfiltfilt(sos, sig)

## Stage 3 — STA/LTA

In [ ]:
sta_n = max(1, int(STA_WINDOW_S * fs))
lta_n = max(1, int(LTA_WINDOW_S * fs))
energy = filtered ** 2
cs = np.concatenate(([0.0], np.cumsum(energy)))

sta = np.array([(cs[i+1] - cs[max(0,i-sta_n+1)]) / (i - max(0,i-sta_n+1) + 1) for i in range(len(filtered))])
lta = np.array([(cs[i+1] - cs[max(0,i-lta_n+1)]) / (i - max(0,i-lta_n+1) + 1) for i in range(len(filtered))])
ratio = np.where(lta > 1e-12, sta / lta, 0.0)

## Stage 4 — Spike detection

In [ ]:
baseline_std = np.std(filtered)
above = (ratio > STA_LTA_THRESH) | (np.abs(filtered) > AMP_SIGMA_THRESH * baseline_std)

spikes, in_spike = [], False
for i, flag in enumerate(above):
    if flag and not in_spike:  start = i; in_spike = True
    elif not flag and in_spike: spikes.append((start, i)); in_spike = False
if in_spike: spikes.append((start, len(above)-1))

gap_n  = int(MERGE_GAP_S * fs)
merged = []
for s, e in spikes:
    if merged and s - merged[-1][1] <= gap_n: merged[-1] = (merged[-1][0], e)
    else: merged.append((s, e))
spikes = merged
print(f"Spike clusters found: {len(spikes)}")

## Stage 5 — Event window

In [ ]:
window = None
if spikes:
    ws, we = spikes[0][0], spikes[-1][1]
    quiet_n, calm = int(QUIET_GUARD_S * fs), 0
    i = we
    while i < len(ratio) and calm < quiet_n:
        calm = calm+1 if ratio[i] < STA_LTA_THRESH else 0
        if calm == 0: we = i
        i += 1
    window = (ws, min(we, len(ratio)-1))

## Stage 6 — Diagram

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Earthquake Detection Report', fontsize=13, fontweight='bold')

axes[0].plot(t_s, mag,      color='#4a90d9', lw=0.6, label='Raw |a|')
axes[0].set_ylabel('Magnitude (g)')
axes[1].plot(t_s, filtered, color='#2ecc71', lw=0.6, label='Filtered')
axes[1].set_ylabel('Filtered (g)')
axes[2].plot(t_s, ratio,    color='#e67e22', lw=0.8, label='STA/LTA')
axes[2].axhline(STA_LTA_THRESH, color='red', ls='--', lw=1, label=f'Threshold ({STA_LTA_THRESH})')
axes[2].set_ylabel('STA/LTA'); axes[2].set_xlabel('Time (s)')

for ax in axes:
    for s, _ in spikes: ax.axvline(t_s[s], color='red', lw=1.0, alpha=0.6)
    if window: ax.axvspan(t_s[window[0]], t_s[window[1]], color='red', alpha=0.12)
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('earthquake_report.png', dpi=150)
plt.show()
print('Diagram saved → earthquake_report.png')

## Stage 7 — Output report

In [ ]:
print('─' * 55)
if window is None:
    print('✗  No earthquake detected in this recording.')
else:
    ws, we = window
    duration = t_s[we] - t_s[ws]
    start_lbl = str(timestamps.iloc[ws]) if timestamps is not None else f'{t_s[ws]:.3f} s'
    end_lbl   = str(timestamps.iloc[we]) if timestamps is not None else f'{t_s[we]:.3f} s'
    print('✓  Earthquake detected!')
    print(f'   Start    : {start_lbl}')
    print(f'   End      : {end_lbl}')
    print(f'   Duration : {duration:.1f} s')
print('─' * 55)